# 🎙️ VoiceBatch Studio v2.1.5 - [Final Error Fix]
यह वर्जन आपके GitHub 'Space' एरर को पूरी तरह फिक्स कर देगा।

In [ ]:
# @title 🔑 Step 1: GitHub & Engine Setup
import os, shutil

GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "" # @param {type:"string"}

# 🛑 SPACE ERROR FIX: यूजरनेम से स्पेस हटाना
GITHUB_USER = GITHUB_USER.strip().replace(" ", "")

if GITHUB_USER and GITHUB_TOKEN and REPO_NAME:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    
    # पुराने फोल्डर को हटाना ताकि नया फ्रेश क्लोन हो सके
    if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
    
    print(f"⏳ {REPO_NAME} को GitHub से लिंक किया जा रहा है...")
    !git clone {REPO_URL}
    
    # फोल्डर के अंदर जाना
    %cd {REPO_NAME}
    os.makedirs("outputs", exist_ok=True)
    
    !pip install -q gradio librosa soundfile coqui-tts
    print(f"✅ सफलतापूर्वक लिंक हो गया! यूजरनेम: {GITHUB_USER}")
else:
    print("⚠️ भाई, टोकन और यूजरनेम भरना ज़रूरी है!")

In [ ]:
# @title 🚀 Step 2: app.py (Expressions + Orange Theme)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def studio_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    
    # [laugh], [sigh] के लिए पॉज़ तैयार करना
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    out_wav = 'outputs/v_batch_pro.wav'
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=out_wav, split_sentences=True)
    
    y, sr = librosa.load(out_wav)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_wav, y, sr)
    return out_wav

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ Master Voice Studio v2.1.5')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Tags: [laugh], [sigh])', lines=8)
            smp = gr.Audio(label='Sample Voice', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.8, 1.2, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-3, 3, 0, step=1, label="Pitch")
            sil = gr.Checkbox(label="Remove Silence", value=True)
            btn = gr.Button('Generate Emotional Voice 🔱', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Final Result')
            gr.Markdown('**Support:** `[laugh]`, `[sigh]`, `[cough]`')

    btn.click(studio_engine, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है!")
!python app.py

In [ ]:
# @title ⬆️ Step 3: GitHub Permanent Save
!git config --global user.email "user@example.com"
!git config --global user.name "{GITHUB_USER}"
!git add .
!git commit -m "Expression Update Fixed"
!git push
print("✅ GitHub पर सफलतापूर्वक सेव हो गया!")